# Extract Skills from Requirements Text

**Purpose:** Extract technical skills from `requirements_text` field and populate `required_skills` as a JSON list

**Process:**
1. Load jobs with requirements_text from database
2. Extract technical skills using pattern matching
3. Store as JSON array in required_skills field
4. Update database

**Run cells in order**

## 1. Install Dependencies

In [1]:
!pip install pymysql sqlalchemy pandas tqdm cryptography requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.9 MB/s eta 0:00:00


## 2. Import Libraries

In [2]:
import pymysql
import pandas as pd
import json
import re
import ssl
import tempfile
import os
import requests
from sqlalchemy import create_engine, text
from tqdm.notebook import tqdm
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries imported successfully')

✅ Libraries imported successfully


## 3. Database Configuration

In [3]:
# ========== DATABASE CONFIG ==========
DB_CONFIG = {
    'host': 'gateway01.ap-southeast-1.prod.aws.tidbcloud.com',
    'port': 4000,
    'user': '4GJhpnEevqoZfyD.root',
    'password': 'oiK2dgnVVJLVHL4v',
    'database': 'data-mining',
    'charset': 'utf8mb4'
}

# ========== SSL CERTIFICATE ==========
CA_CERT_CONTENT = """
-----BEGIN CERTIFICATE-----
MIIFazCCA1OgAwIBAgIRAIIQz7DSQONZRGPgu2OCiwAwDQYJKoZIhvcNAQELBQAw
TzELMAkGA1UEBhMCVVMxKTAnBgNVBAoTIEludGVybmV0IFNlY3VyaXR5IFJlc2Vh
cmNoIEdyb3VwMRUwEwYDVQQDEwxJU1JHIFJvb3QgWDEwHhcNMTUwNjA0MTEwNDM4
WhcNMzUwNjA0MTEwNDM4WjBPMQswCQYDVQQGEwJVUzEpMCcGA1UEChMgSW50ZXJu
ZXQgU2VjdXJpdHkgUmVzZWFyY2ggR3JvdXAxFTATBgNVBAMTDElTUkcgUm9vdCBY
MTCCAiIwDQYJKoZIhvcNAQEBBQADggIPADCCAgoCggIBAK3oJHP0FDfzm54rVygc
h77ct984kIxuPOZXoHj3dcKi/vVqbvYATyjb3miGbESTtrFj/RQSa78f0uoxmyF+
0TM8ukj13Xnfs7j/EvEhmkvBioZxaUpmZmyPfjxwv60pIgbz5MDmgK7iS4+3mX6U
A5/TR5d8mUgjU+g4rk8Kb4Mu0UlXjIB0ttov0DiNewNwIRt18jA8+o+u3dpjq+sW
T8KOEUt+zwvo/7V3LvSye0rgTBIlDHCNAymg4VMk7BPZ7hm/ELNKjD+Jo2FR3qyH
B5T0Y3HsLuJvW5iB4YlcNHlsdu87kGJ55tukmi8mxdAQ4Q7e2RCOFvu396j3x+UC
B5iPNgiV5+I3lg02dZ77DnKxHZu8A/lJBdiB3QW0KtZB6awBdpUKD9jf1b0SHzUv
KBds0pjBqAlkd25HN7rOrFleaJ1/ctaJxQZBKT5ZPt0m9STJEadao0xAH0ahmbWn
OlFuhjuefXKnEgV4We0+UXgVCwOPjdAvBbI+e0ocS3MFEvzG6uBQE3xDk3SzynTn
jh8BCNAw1FtxNrQHusEwMFxIt4I7mKZ9YIqioymCzLq9gwQbooMDQaHWBfEbwrbw
qHyGO0aoSCqI3Haadr8faqU9GY/rOPNk3sgrDQoo//fb4hVC1CLQJ13hef4Y53CI
rU7m2Ys6xt0nUW7/vGT1M0NPAgMBAAGjQjBAMA4GA1UdDwEB/wQEAwIBBjAPBgNV
HRMBAf8EBTADAQH/MB0GA1UdDgQWBBR5tFnme7bl5AFzgAiIyBpY9umbbjANBgkq
hkiG9w0BAQsFAAOCAgEAVR9YqbyyqFDQDLHYGmkgJykIrGF1XIpu+ILlaS/V9lZL
ubhzEFnTIZd+50xx+7LSYK05qAvqFyFWhfFQDlnrzuBZ6brJFe+GnY+EgPbk6ZGQ
3BebYhtF8GaV0nxvwuo77x/Py9auJ/GpsMiu/X1+mvoiBOv/2X/qkSsisRcOj/KK
NFtY2PwByVS5uCbMiogziUwthDyC3+6WVwW6LLv3xLfHTjuCvjHIInNzktHCgKQ5
ORAzI4JMPJ+GslWYHb4phowim57iaztXOoJwTdwJx4nLCgdNbOhdjsnvzqvHu7Ur
TkXWStAmzOVyyghqpZXjFaH3pO3JLF+l+/+sKAIuvtd7u+Nxe5AW0wdeRlN8NwdC
jNPElpzVmbUq4JUagEiuTDkHzsxHpFKVK7q4+63SM1N95R1NbdWhscdCb+ZAJzVc
oyi3B43njTOQ5yOf+1CceWxG1bQVs5ZufpsMljq4Ui0/1lvh+wjChP4kqKOJ2qxq
4RgqsahDYVvTH9w7jXbyLeiNdd8XM2w9U/t7y0Ff/9yi0GE44Za4rF2LN9d11TPA
mRGunUHBcnWEvgJBQl9nJEiU0Zsnvgc/ubhPgXRR4Xq37Z0j4r7g1SgEEzwxA57d
emyPxgcYxn/eR44/KJ4EBs+lVDR3veyJm+kXQ99b21/+jh5Xos1AnX5iItreGCc=
-----END CERTIFICATE-----
""".strip()

print(f'📊 Database: {DB_CONFIG["database"]}')
print(f'🔗 Host: {DB_CONFIG["host"]}:{DB_CONFIG["port"]}')
print(f'📜 SSL configured: ✅')

📊 Database: data-mining
🔗 Host: gateway01.ap-southeast-1.prod.aws.tidbcloud.com:4000
📜 SSL configured: ✅


## 4. Setup Database Connection

In [4]:
temp_files = []
try:
    ssl_context = ssl.create_default_context()
    
    if CA_CERT_CONTENT:
        ca_temp = tempfile.NamedTemporaryFile(mode='w', suffix='.pem', delete=False)
        ca_temp.write(CA_CERT_CONTENT)
        ca_temp.close()
        temp_files.append(ca_temp.name)
        ssl_context.load_verify_locations(ca_temp.name)
        print('✅ Loaded CA certificate')
    
    DATABASE_URL = f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}?charset={DB_CONFIG['charset']}"
    engine = create_engine(DATABASE_URL, connect_args={'ssl': ssl_context}, pool_pre_ping=True)
    
    print('\n🔄 Testing connection...')
    with engine.connect() as conn:
        result = conn.execute(text('SELECT COUNT(*) FROM jobs'))
        total = result.fetchone()[0]
        print(f'\n✅ Connection successful!')
        print(f'📊 Total jobs: {total:,}')
        
except Exception as e:
    print(f'❌ Error: {e}')
    for f in temp_files:
        try: os.unlink(f)
        except: pass
    raise

✅ Loaded CA certificate

🔄 Testing connection...

✅ Connection successful!
📊 Total jobs: 12,117


## 5. Load Data from Database

In [5]:
print('📥 Loading jobs data...')
query = '''
SELECT 
    id, 
    title,
    requirements_text,
    required_skills,
    description,
    company_name
FROM jobs 
WHERE requirements_text IS NOT NULL 
    AND requirements_text != ''
ORDER BY id
'''

df = pd.read_sql(query, engine)
print(f'✅ Loaded {len(df):,} jobs with requirements_text')

# Show current state
empty_skills = df['required_skills'].isna().sum()
has_skills = len(df) - empty_skills
print(f'\n📊 Current state:')
print(f'  required_skills filled: {has_skills:,} ({has_skills/len(df)*100:.1f}%)')
print(f'  required_skills empty: {empty_skills:,} ({empty_skills/len(df)*100:.1f}%)')

df.head(3)

📥 Loading jobs data...
✅ Loaded 10,044 jobs with requirements_text

📊 Current state:
  required_skills filled: 9,373 (93.3%)
  required_skills empty: 671 (6.7%)


,id,title,requirements_text,required_skills,description,company_name
0,3462,Frontend developer (PA project),Your skills & qualifications:\n\nExperience in...,"[""CSS"", ""HTML"", ""Git"", ""Restful Api"", ""SASS"", ...",Your role & responsibilities:\nWe’re looking f...,CUBICASA
1,3470,TIGER TRIBE – SENIOR BACKEND DEVELOPER,Your skills & qualifications:\n\n5+ years of e...,"[""ASP.NET"", ""C#"", "".NET"", ""SQL Server"", ""Restf...","Your role & responsibilities:\nDesign, develop...",TIGER TRIBE
2,3474,Frontend developer (PA project),Your skills & qualifications:\n\nExperience in...,"[""CSS"", ""HTML"", ""Git"", ""Restful Api"", ""SASS"", ...",Your role & responsibilities:\nWe’re looking f...,CUBICASA


In [6]:
class LightcastSkillsLoader:
    """
    Loads comprehensive skill taxonomy using:
    1. Lightcast Open Skills API (if available)
    2. Local comprehensive skill dictionary (fallback)
    """
    
    def __init__(self):
        self.skills_db = set()
        self.skill_categories = {}
        
    def load_from_lightcast_api(self, api_key=None):
        """
        Load skills from Lightcast Open Skills API
        API Docs: https://docs.lightcast.dev/apis/skills
        
        NOTE: Lightcast Open Skills API is FREE for non-commercial use.
        Get your API key at: https://lightcast.io/open-skills
        """
        if not api_key:
            print('⚠️  No Lightcast API key provided. Using local dictionary.')
            return False
        
        try:
            print('🔄 Fetching skills from Lightcast API...')
            
            # Lightcast API endpoint for skills list
            url = 'https://emsiservices.com/skills/versions/latest/skills'
            headers = {
                'Authorization': f'Bearer {api_key}',
                'Content-Type': 'application/json'
            }
            
            response = requests.get(url, headers=headers, timeout=30)
            
            if response.status_code == 200:
                data = response.json()
                skills_data = data.get('data', [])
                
                for skill in skills_data:
                    skill_name = skill.get('name')
                    skill_type = skill.get('type', {}).get('name', '')
                    
                    if skill_name:
                        self.skills_db.add(skill_name)
                        
                        # Categorize by type
                        if skill_type not in self.skill_categories:
                            self.skill_categories[skill_type] = []
                        self.skill_categories[skill_type].append(skill_name)
                
                print(f'✅ Loaded {len(self.skills_db):,} skills from Lightcast API')
                print(f'📊 Categories: {len(self.skill_categories)}')
                return True
            else:
                print(f'⚠️  API request failed: {response.status_code}')
                return False
                
        except Exception as e:
            print(f'⚠️  Error loading from Lightcast API: {str(e)[:100]}')
            return False
    
    def load_local_comprehensive_dictionary(self):
        """
        Load comprehensive local skill dictionary
        Based on industry standards and common job requirements
        """
        print('📚 Loading comprehensive local skill dictionary...')
        
        # Comprehensive skill dictionary (700+ skills)
        local_skills = {
            # Programming Languages (50+)
            'Python', 'Java', 'JavaScript', 'TypeScript', 'C++', 'C#', 'C', 'Go', 'Rust', 
            'Ruby', 'PHP', 'Swift', 'Kotlin', 'Scala', 'R', 'Dart', 'Objective-C',
            'Perl', 'Shell', 'Bash', 'PowerShell', 'VBA', 'MATLAB', 'Groovy', 'Haskell',
            'Clojure', 'Elixir', 'Erlang', 'F#', 'Julia', 'Lua', 'Assembly', 'COBOL',
            'Fortran', 'Ada', 'Prolog', 'Lisp', 'Scheme', 'OCaml', 'Racket', 'Crystal',
            'Nim', 'Zig', 'D', 'Hack', 'Pascal', 'Delphi', 'ActionScript', 'CoffeeScript',
            
            # Web Frontend (80+)
            'React', 'Angular', 'Vue.js', 'Vue', 'Next.js', 'Nuxt.js', 'Svelte', 'SvelteKit',
            'jQuery', 'HTML', 'HTML5', 'CSS', 'CSS3', 'SASS', 'SCSS', 'LESS', 'Stylus',
            'Bootstrap', 'Tailwind CSS', 'Material-UI', 'Ant Design', 'Chakra UI',
            'Webpack', 'Vite', 'Rollup', 'Parcel', 'Redux', 'MobX', 'Vuex', 'Pinia',
            'RxJS', 'Axios', 'Fetch API', 'Web Components', 'Lit', 'Stencil',
            'Ember.js', 'Backbone.js', 'Knockout.js', 'Preact', 'Solid.js', 'Qwik',
            'Alpine.js', 'HTMX', 'Hotwire', 'Stimulus', 'Turbo', 'Web Animations API',
            'Canvas API', 'WebGL', 'Three.js', 'D3.js', 'Chart.js', 'Highcharts',
            'Recharts', 'Victory', 'Plotly', 'Babylon.js', 'PixiJS', 'Phaser',
            'Gatsby', 'Astro', 'Remix', 'SolidStart', 'Fresh', 'Marko',
            'GraphQL', 'Apollo Client', 'Relay', 'urql', 'tRPC', 'React Query',
            'SWR', 'Zustand', 'Jotai', 'Recoil', 'Immer', 'Zod', 'Yup',
            
            # Backend Frameworks (60+)
            'Django', 'Flask', 'FastAPI', 'Spring Boot', 'Spring', 'Spring Cloud',
            'Express.js', 'Express', 'Node.js', 'NestJS', 'Koa', 'Hapi', 'Fastify',
            'Rails', 'Ruby on Rails', 'Sinatra', 'Laravel', 'Symfony', 'CodeIgniter',
            'ASP.NET', '.NET Core', '.NET', 'Entity Framework', 'Dapper',
            'Gin', 'Echo', 'Fiber', 'Beego', 'Revel', 'Iris',
            'Actix', 'Rocket', 'Axum', 'Warp', 'Tide',
            'Phoenix', 'Plug', 'Cowboy', 'Ecto',
            'Play Framework', 'Akka', 'Vert.x', 'Quarkus', 'Micronaut', 'Helidon',
            'Dropwizard', 'Spark Java', 'Javalin', 'Struts', 'JSF', 'Grails',
            'CakePHP', 'Yii', 'Zend', 'Slim', 'Lumen', 'Phalcon',
            'Tornado', 'Bottle', 'Pyramid', 'CherryPy', 'Sanic', 'Starlette',
            
            # Mobile Development (40+)
            'React Native', 'Flutter', 'Ionic', 'Xamarin', 'Android', 'iOS',
            'SwiftUI', 'UIKit', 'Jetpack Compose', 'Android SDK', 'iOS SDK',
            'Kotlin Multiplatform', 'Capacitor', 'Cordova', 'PhoneGap',
            'NativeScript', 'Expo', 'React Navigation', 'Flutter Bloc', 'Provider',
            'GetX', 'Riverpod', 'Android Studio', 'Xcode', 'CocoaPods',
            'Gradle', 'Maven', 'Fastlane', 'TestFlight', 'Firebase',
            'Room', 'Realm', 'Core Data', 'SQLite', 'Hive', 'ObjectBox',
            'ARKit', 'ARCore', 'Metal', 'OpenGL ES', 'Vulkan',
            
            # Databases & Data Storage (70+)
            'MySQL', 'PostgreSQL', 'MongoDB', 'Redis', 'Oracle', 'SQL Server', 
            'SQLite', 'MariaDB', 'Cassandra', 'DynamoDB', 'Elasticsearch', 
            'CouchDB', 'Neo4j', 'InfluxDB', 'Firebase', 'Firestore', 'Supabase',
            'TimescaleDB', 'CockroachDB', 'RethinkDB', 'ArangoDB', 'OrientDB',
            'Couchbase', 'HBase', 'Bigtable', 'ScyllaDB', 'Fauna',
            'PlanetScale', 'Neon', 'Turso', 'EdgeDB', 'SurrealDB',
            'Prisma', 'TypeORM', 'Sequelize', 'Mongoose', 'Drizzle ORM',
            'Hibernate', 'MyBatis', 'JOOQ', 'Exposed', 'Slick',
            'SQLAlchemy', 'Peewee', 'Pony ORM', 'Tortoise ORM',
            'ActiveRecord', 'DataMapper', 'Doctrine', 'Eloquent',
            'Knex.js', 'Bookshelf.js', 'Objection.js', 'MikroORM',
            'SQL', 'NoSQL', 'PL/SQL', 'T-SQL', 'PostgreSQL PL/pgSQL',
            'Database Design', 'Data Modeling', 'Query Optimization',
            'Database Administration', 'Database Migration', 'Database Backup',
            'Replication', 'Sharding', 'Partitioning', 'Indexing',
            
            # Cloud Platforms (50+)
            'AWS', 'Azure', 'GCP', 'Google Cloud', 'Google Cloud Platform',
            'Amazon Web Services', 'Microsoft Azure', 'Heroku', 'DigitalOcean', 
            'Vercel', 'Netlify', 'CloudFlare', 'Cloudflare Workers', 'Railway',
            'Render', 'Fly.io', 'AWS Lambda', 'AWS EC2', 'AWS S3', 'AWS RDS',
            'AWS ECS', 'AWS EKS', 'AWS CloudFormation', 'AWS CloudWatch',
            'AWS IAM', 'AWS VPC', 'AWS Route 53', 'AWS API Gateway',
            'Azure Functions', 'Azure App Service', 'Azure SQL', 'Azure Cosmos DB',
            'Azure DevOps', 'Azure Kubernetes Service', 'Azure Storage',
            'Google Compute Engine', 'Google Kubernetes Engine', 'Google Cloud Functions',
            'Google Cloud Storage', 'Google Cloud SQL', 'Google BigQuery',
            'Cloud Architecture', 'Cloud Migration', 'Cloud Security',
            'Multi-Cloud', 'Hybrid Cloud', 'Serverless', 'Edge Computing',
            
            # DevOps & Infrastructure (90+)
            'Docker', 'Kubernetes', 'Jenkins', 'GitLab CI', 'GitHub Actions', 
            'CircleCI', 'Travis CI', 'Terraform', 'Ansible', 'Chef', 'Puppet',
            'Helm', 'ArgoCD', 'Flux', 'Spinnaker', 'Tekton',
            'Prometheus', 'Grafana', 'ELK Stack', 'Datadog', 'New Relic', 
            'Splunk', 'Dynatrace', 'AppDynamics', 'PagerDuty',
            'Nginx', 'Apache', 'HAProxy', 'Traefik', 'Envoy', 'Istio', 'Linkerd',
            'Consul', 'Vault', 'etcd', 'Zookeeper',
            'CI/CD', 'Continuous Integration', 'Continuous Deployment',
            'Infrastructure as Code', 'Configuration Management',
            'Container Orchestration', 'Service Mesh', 'Observability',
            'Monitoring', 'Logging', 'Tracing', 'APM', 'Log Aggregation',
            'Vagrant', 'Packer', 'VMware', 'VirtualBox', 'Hyper-V',
            'OpenShift', 'Rancher', 'Nomad', 'Docker Swarm', 'Podman',
            'containerd', 'CRI-O', 'BuildKit', 'Kaniko', 'Jib',
            'Pulumi', 'CloudFormation', 'ARM Templates', 'Crossplane',
            'SaltStack', 'CFEngine', 'Capistrano', 'Fabric',
            'Nagios', 'Zabbix', 'Icinga', 'Sensu', 'Telegraf',
            'Fluentd', 'Logstash', 'Filebeat', 'Vector', 'Loki',
            'Jaeger', 'Zipkin', 'OpenTelemetry', 'Sentry', 'Rollbar',
            
            # Data Science & ML (100+)
            'TensorFlow', 'PyTorch', 'Keras', 'Scikit-learn', 'Pandas', 'NumPy',
            'Matplotlib', 'Seaborn', 'Plotly', 'Bokeh', 'Altair',
            'Jupyter', 'JupyterLab', 'Google Colab', 'Kaggle',
            'Apache Spark', 'PySpark', 'Hadoop', 'Hive', 'Pig', 'HBase',
            'Airflow', 'Prefect', 'Dagster', 'Luigi', 'MLflow', 'Kubeflow',
            'ONNX', 'XGBoost', 'LightGBM', 'CatBoost', 'H2O.ai',
            'OpenCV', 'Pillow', 'scikit-image', 'SimpleCV',
            'NLTK', 'spaCy', 'Gensim', 'TextBlob', 'Transformers', 'Hugging Face',
            'LangChain', 'LlamaIndex', 'Anthropic', 'OpenAI API',
            'Stable Diffusion', 'DALL-E', 'Midjourney', 'GPT', 'BERT',
            'Machine Learning', 'Deep Learning', 'Neural Networks',
            'Computer Vision', 'Natural Language Processing', 'NLP',
            'Reinforcement Learning', 'Supervised Learning', 'Unsupervised Learning',
            'Feature Engineering', 'Model Training', 'Model Deployment',
            'A/B Testing', 'Statistical Analysis', 'Data Analysis',
            'Data Visualization', 'Data Mining', 'Data Wrangling',
            'Time Series Analysis', 'Forecasting', 'Anomaly Detection',
            'Recommendation Systems', 'Clustering', 'Classification',
            'Regression', 'Dimensionality Reduction', 'Ensemble Methods',
            'Gradient Boosting', 'Random Forest', 'Decision Trees',
            'SVM', 'K-Means', 'PCA', 't-SNE', 'UMAP',
            'Dask', 'Ray', 'Modin', 'Vaex', 'Polars',
            'DVC', 'Weights & Biases', 'Neptune.ai', 'ClearML',
            'SageMaker', 'Azure ML', 'Vertex AI', 'Databricks',
            'Snowflake', 'BigQuery', 'Redshift', 'Athena', 'Presto', 'Trino',
            
            # Message Queues & Streaming (30+)
            'Kafka', 'Apache Kafka', 'RabbitMQ', 'ActiveMQ', 'Redis Pub/Sub', 
            'AWS SQS', 'AWS SNS', 'Azure Service Bus', 'Azure Event Hubs',
            'Google Pub/Sub', 'Pulsar', 'NATS', 'MQTT', 'ZeroMQ',
            'Event Sourcing', 'CQRS', 'Message Queue', 'Event Streaming',
            'Kinesis', 'Flink', 'Storm', 'Samza', 'NiFi',
            'Debezium', 'Maxwell', 'Canal', 'Kafka Connect', 'Kafka Streams',
            
            # Testing & QA (50+)
            'Jest', 'Mocha', 'Chai', 'Jasmine', 'Karma', 'Vitest',
            'Cypress', 'Selenium', 'Playwright', 'Puppeteer', 'WebDriver',
            'JUnit', 'TestNG', 'Mockito', 'JMock', 'PowerMock',
            'PyTest', 'unittest', 'nose', 'Hypothesis', 'Behave',
            'RSpec', 'Cucumber', 'SpecFlow', 'Gherkin',
            'Postman', 'Insomnia', 'REST Client', 'HTTPie',
            'JMeter', 'Gatling', 'Locust', 'K6', 'Artillery',
            'Unit Testing', 'Integration Testing', 'End-to-End Testing',
            'Performance Testing', 'Load Testing', 'Stress Testing',
            'Test Automation', 'Test-Driven Development', 'TDD', 'BDD',
            'API Testing', 'UI Testing', 'Regression Testing',
            'SonarQube', 'ESLint', 'Prettier', 'Black', 'Flake8', 'Pylint',
            
            # Version Control (15+)
            'Git', 'GitHub', 'GitLab', 'Bitbucket', 'SVN', 'Mercurial',
            'Git Flow', 'GitHub Flow', 'Trunk-Based Development',
            'Code Review', 'Pull Request', 'Merge Request',
            'Version Control', 'Source Control', 'Branch Management',
            
            # API & Integration (30+)
            'REST API', 'RESTful API', 'GraphQL', 'gRPC', 'SOAP', 'WebSocket',
            'OAuth', 'OAuth 2.0', 'JWT', 'SAML', 'OpenID Connect',
            'API Gateway', 'API Design', 'API Development',
            'Postman', 'Swagger', 'OpenAPI', 'API Documentation',
            'Webhooks', 'Server-Sent Events', 'Long Polling',
            'JSON', 'XML', 'Protocol Buffers', 'Avro', 'Thrift',
            'API Security', 'Rate Limiting', 'API Versioning',
            'Microservices', 'Service-Oriented Architecture', 'SOA',
            
            # Security (40+)
            'OWASP', 'Penetration Testing', 'Ethical Hacking',
            'Burp Suite', 'Metasploit', 'Wireshark', 'Nmap', 'Nessus',
            'SSL/TLS', 'Encryption', 'Cryptography', 'PKI',
            'Active Directory', 'LDAP', 'Single Sign-On', 'SSO',
            'Multi-Factor Authentication', 'MFA', '2FA',
            'Security Auditing', 'Vulnerability Assessment', 'Threat Modeling',
            'Incident Response', 'Forensics', 'SIEM', 'IDS/IPS',
            'Firewall', 'WAF', 'DDoS Protection', 'Network Security',
            'Application Security', 'Cloud Security', 'Container Security',
            'Zero Trust', 'DevSecOps', 'Security Scanning',
            'Static Analysis', 'Dynamic Analysis', 'SAST', 'DAST',
            'Dependency Scanning', 'Secrets Management',
            
            # Project Management & Methodologies (40+)
            'Agile', 'Scrum', 'Kanban', 'Lean', 'SAFe', 'XP',
            'DevOps', 'DevSecOps', 'GitOps', 'DataOps', 'MLOps', 'AIOps',
            'Jira', 'Confluence', 'Trello', 'Asana', 'Monday.com',
            'Slack', 'Microsoft Teams', 'Discord', 'Zoom', 'Miro',
            'Waterfall', 'V-Model', 'Spiral Model', 'Iterative',
            'Pair Programming', 'Mob Programming', 'Code Review',
            'Sprint Planning', 'Retrospectives', 'Daily Standups',
            'Backlog Grooming', 'User Stories', 'Story Points',
            'Burndown Charts', 'Velocity Tracking',
            'Project Management', 'Product Management', 'Program Management',
            'Stakeholder Management', 'Risk Management', 'Change Management',
            
            # Design & UX (40+)
            'Figma', 'Sketch', 'Adobe XD', 'InVision', 'Zeplin', 'Abstract',
            'Photoshop', 'Illustrator', 'After Effects', 'Premiere Pro',
            'UI Design', 'UX Design', 'Interaction Design', 'Visual Design',
            'User Research', 'Usability Testing', 'A/B Testing',
            'Wireframing', 'Prototyping', 'Mockups', 'Design Systems',
            'Responsive Design', 'Mobile-First Design', 'Accessibility',
            'WCAG', 'ARIA', 'Color Theory', 'Typography',
            'Information Architecture', 'User Flow', 'Customer Journey',
            'Persona Development', 'Design Thinking', 'Human-Centered Design',
            'Atomic Design', 'Material Design', 'Flat Design', 'Minimalism',
            'Motion Design', 'Micro-interactions', 'Animation',
            
            # Blockchain & Web3 (30+)
            'Solidity', 'Ethereum', 'Web3.js', 'Ethers.js', 'Hardhat', 'Truffle',
            'Smart Contracts', 'DApp', 'Bitcoin', 'Blockchain',
            'Hyperledger', 'Fabric', 'Corda', 'Quorum',
            'NFT', 'DeFi', 'DAO', 'Token', 'Cryptocurrency',
            'Polygon', 'Binance Smart Chain', 'Avalanche', 'Solana',
            'IPFS', 'Filecoin', 'The Graph', 'Chainlink',
            'MetaMask', 'WalletConnect', 'Alchemy', 'Infura', 'Moralis',
            
            # Game Development (25+)
            'Unity', 'Unreal Engine', 'Godot', 'GameMaker', 'Construct',
            'Cocos2d', 'LibGDX', 'MonoGame', 'Pygame', 'Panda3D',
            'Phaser', 'PixiJS', 'Babylon.js', 'PlayCanvas',
            'C++ for Games', 'C# for Games', 'Game Design', 'Level Design',
            'Game Physics', 'AI for Games', 'Shader Programming',
            'OpenGL', 'DirectX', 'Vulkan', 'Metal',
            
            # Business Intelligence (20+)
            'Tableau', 'Power BI', 'Looker', 'Metabase', 'Superset',
            'QuickSight', 'QlikView', 'Qlik Sense', 'MicroStrategy',
            'Business Intelligence', 'Data Warehousing', 'ETL',
            'OLAP', 'Data Cube', 'Dimensional Modeling',
            'Star Schema', 'Snowflake Schema', 'Fact Tables',
            'Data Pipeline', 'Data Integration',
            
            # CMS & E-commerce (25+)
            'WordPress', 'Drupal', 'Joomla', 'Contentful', 'Strapi', 'Sanity',
            'Ghost', 'Hugo', 'Jekyll', 'Eleventy', '11ty',
            'Shopify', 'Magento', 'WooCommerce', 'BigCommerce', 'PrestaShop',
            'Salesforce Commerce Cloud', 'Adobe Commerce',
            'Headless CMS', 'Jamstack', 'Static Site Generation',
            'Content Management', 'E-commerce Development',
            'Payment Integration', 'Stripe', 'PayPal',
            
            # Soft Skills & Leadership (30+)
            'Problem Solving', 'Critical Thinking', 'Analytical Skills',
            'Team Leadership', 'Team Management', 'People Management',
            'Communication', 'Written Communication', 'Verbal Communication',
            'Presentation Skills', 'Public Speaking',
            'Code Review', 'Technical Writing', 'Documentation',
            'Mentoring', 'Coaching', 'Training', 'Teaching',
            'Architecture Design', 'System Design', 'Solution Architecture',
            'Technical Leadership', 'Cross-Functional Collaboration',
            'Stakeholder Communication', 'Client Management',
            'Conflict Resolution', 'Negotiation', 'Time Management',
            'Decision Making', 'Strategic Thinking',
        }
        
        self.skills_db = local_skills
        print(f'✅ Loaded {len(self.skills_db):,} skills from local dictionary')
        return True
    
    def load(self, lightcast_api_key=None):
        """
        Load skills from Lightcast API if available, otherwise use local dictionary
        """
        # Try Lightcast API first
        if lightcast_api_key:
            success = self.load_from_lightcast_api(lightcast_api_key)
            if success:
                return self.skills_db
        
        # Fallback to local dictionary
        self.load_local_comprehensive_dictionary()
        return self.skills_db


# ========== CONFIGURATION ==========
# Set your Lightcast API key here (optional)
# Get free API key at: https://lightcast.io/open-skills
LIGHTCAST_API_KEY = None  # or 'your-api-key-here'

# Load skills
print('🚀 Loading skills dictionary...')
print('='*80)
loader = LightcastSkillsLoader()
SKILLS_DATABASE = loader.load(LIGHTCAST_API_KEY)

print('='*80)
print(f'📊 Total skills available: {len(SKILLS_DATABASE):,}')
print()
print('💡 TIP: For even more comprehensive results, get a FREE Lightcast API key at:')
print('   https://lightcast.io/open-skills')
print('   Then update LIGHTCAST_API_KEY variable above.')
print('='*80)

🚀 Loading skills dictionary...
📚 Loading comprehensive local skill dictionary...
✅ Loaded 910 skills from local dictionary
📊 Total skills available: 910

💡 TIP: For even more comprehensive results, get a FREE Lightcast API key at:
   https://lightcast.io/open-skills
   Then update LIGHTCAST_API_KEY variable above.


## 5.5 Load Skills Dictionary (Lightcast API or Local)

## 6. Define Skill Extraction Class

In [7]:
class SkillExtractor:
    def __init__(self, skills_database):
        """
        Initialize with a comprehensive skills database
        
        Args:
            skills_database: Set or list of skill names to extract
        """
        # Use provided skills database
        self.skills_db = set(skills_database) if not isinstance(skills_database, set) else skills_database
        
        # Create case-insensitive lookup
        self.skills_lower = {skill.lower(): skill for skill in self.skills_db}
        
        # Compile regex patterns for better matching
        self.skill_patterns = {}
        for skill in self.skills_db:
            # Create word boundary pattern for exact matches
            # Escape special regex characters
            escaped_skill = re.escape(skill)
            pattern = r'\b' + escaped_skill + r'\b'
            try:
                self.skill_patterns[skill] = re.compile(pattern, re.IGNORECASE)
            except:
                # If regex compilation fails, skip this skill
                continue
        
        self.stats = {
            'processed': 0,
            'skills_found': 0,
            'empty_results': 0,
            'avg_skills_per_job': 0
        }
    
    def extract_skills(self, text):
        """Extract skills from text using pattern matching"""
        if not text or not str(text).strip():
            return []
        
        text = str(text)
        found_skills = set()
        
        # Search for each skill pattern in text
        for skill, pattern in self.skill_patterns.items():
            if pattern.search(text):
                found_skills.add(skill)
        
        # Return sorted list for consistency
        return sorted(list(found_skills))
    
    def extract_from_job(self, row):
        """Extract skills from job requirements_text and description"""
        requirements_text = row.get('requirements_text', '')
        description = row.get('description', '')
        
        # Combine texts for comprehensive extraction
        combined_text = f"{requirements_text} {description}"
        
        skills = self.extract_skills(combined_text)
        
        self.stats['processed'] += 1
        if skills:
            self.stats['skills_found'] += len(skills)
        else:
            self.stats['empty_results'] += 1
        
        return skills
    
    def print_stats(self):
        """Print extraction statistics"""
        if self.stats['processed'] > 0:
            avg = self.stats['skills_found'] / self.stats['processed']
            self.stats['avg_skills_per_job'] = avg
        
        print(f"📊 EXTRACTION STATISTICS:")
        print(f"  Processed jobs: {self.stats['processed']:,}")
        print(f"  Total skills found: {self.stats['skills_found']:,}")
        print(f"  Jobs with no skills: {self.stats['empty_results']:,}")
        print(f"  Average skills per job: {self.stats['avg_skills_per_job']:.2f}")

print('✅ SkillExtractor class defined')
print(f'📚 Will use loaded skills database: {len(SKILLS_DATABASE):,} skills')
print('🔍 Extraction method: Pattern matching with word boundaries')
print('📋 Sources: requirements_text + description (combined)')

✅ SkillExtractor class defined
📚 Will use loaded skills database: 910 skills
🔍 Extraction method: Pattern matching with word boundaries
📋 Sources: requirements_text + description (combined)


## 7. Extract Skills from All Jobs

In [8]:
import time

# Initialize extractor with loaded skills database
extractor = SkillExtractor(SKILLS_DATABASE)
df_updated = df.copy()

print(f'🚀 Starting skill extraction for {len(df):,} jobs')
print(f'🔥 Mode: FULL EXTRACTION - Will update ALL jobs')
print(f'📚 Using {len(SKILLS_DATABASE):,} skills from dictionary')
print(f'⏱️  Estimated time: ~{len(df)/500:.1f} seconds\n')

start_time = time.time()
updated_count = 0

for idx in tqdm(range(len(df_updated)), desc='Extracting skills'):
    row = df_updated.iloc[idx]
    
    # Extract skills
    skills = extractor.extract_from_job(row)
    
    # Update required_skills as JSON
    if skills:
        df_updated.at[idx, 'required_skills'] = json.dumps(skills)
        updated_count += 1
    else:
        # Store empty array if no skills found
        df_updated.at[idx, 'required_skills'] = json.dumps([])
    
    # Progress reporting every 1000 jobs
    if (idx + 1) % 1000 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(df_updated) - idx - 1) / rate
        print(f'\n[{idx+1}/{len(df_updated)}] Time: {elapsed:.1f}s | '
              f'Remaining: {remaining:.1f}s | Rate: {rate:.1f} jobs/s')
        extractor.print_stats()

elapsed_time = time.time() - start_time
print(f'\n✅ Extraction complete!')
print(f'⏱️  Total time: {elapsed_time:.1f} seconds')
print(f'⚡ Average rate: {len(df)/elapsed_time:.1f} jobs/second')
print(f'\n📊 FINAL STATISTICS:')
extractor.print_stats()
print(f'📝 Jobs with skills extracted: {updated_count:,} ({updated_count/len(df)*100:.1f}%)')

🚀 Starting skill extraction for 10,044 jobs
🔥 Mode: FULL EXTRACTION - Will update ALL jobs
📚 Using 910 skills from dictionary
⏱️  Estimated time: ~20.1 seconds



Extracting skills:   0%|          | 0/10044 [00:00<?, ?it/s]


[1000/10044] Time: 80.1s | Remaining: 724.6s | Rate: 12.5 jobs/s
📊 EXTRACTION STATISTICS:
  Processed jobs: 1,000
  Total skills found: 12,233
  Jobs with no skills: 20
  Average skills per job: 12.23

[2000/10044] Time: 109.5s | Remaining: 440.3s | Rate: 18.3 jobs/s
📊 EXTRACTION STATISTICS:
  Processed jobs: 2,000
  Total skills found: 20,106
  Jobs with no skills: 63
  Average skills per job: 10.05

[3000/10044] Time: 157.0s | Remaining: 368.5s | Rate: 19.1 jobs/s
📊 EXTRACTION STATISTICS:
  Processed jobs: 3,000
  Total skills found: 28,409
  Jobs with no skills: 102
  Average skills per job: 9.47

[4000/10044] Time: 319.4s | Remaining: 482.6s | Rate: 12.5 jobs/s
📊 EXTRACTION STATISTICS:
  Processed jobs: 4,000
  Total skills found: 41,711
  Jobs with no skills: 103
  Average skills per job: 10.43

[5000/10044] Time: 464.8s | Remaining: 468.9s | Rate: 10.8 jobs/s
📊 EXTRACTION STATISTICS:
  Processed jobs: 5,000
  Total skills found: 53,675
  Jobs with no skills: 111
  Average skills

## 8. Preview Extracted Skills

In [9]:
print('🔍 SAMPLE SKILL EXTRACTIONS (First 15 jobs with skills)\n')
print('='*100)

shown = 0
for idx in range(len(df_updated)):
    row = df_updated.iloc[idx]
    
    try:
        skills = json.loads(row['required_skills']) if pd.notna(row['required_skills']) else []
    except:
        skills = []
    
    if skills:
        print(f"\n📋 Job ID: {row['id']}")
        print(f"   Title: {row['title']}")
        print(f"   Company: {row['company_name']}")
        print(f"   🛠️  Skills ({len(skills)}): {', '.join(skills)}")
        
        # Show excerpt from requirements_text
        if pd.notna(row['requirements_text']):
            req_excerpt = str(row['requirements_text'])[:150]
            print(f"   📄 Requirements excerpt: {req_excerpt}...")
        
        shown += 1
        if shown >= 15:
            break

print('\n' + '='*100)

# Show most common skills
print('\n📊 TOP 20 MOST COMMON SKILLS:\n')
all_skills = []
for idx in range(len(df_updated)):
    try:
        skills = json.loads(df_updated.iloc[idx]['required_skills'])
        all_skills.extend(skills)
    except:
        pass

skill_counts = Counter(all_skills)
for skill, count in skill_counts.most_common(20):
    percentage = count / len(df) * 100
    print(f"  {skill}: {count:,} jobs ({percentage:.1f}%)")

🔍 SAMPLE SKILL EXTRACTIONS (First 15 jobs with skills)


📋 Job ID: 3462
   Title: Frontend developer (PA project)
   Company: CUBICASA
   🛠️  Skills (14): CSS, Code Review, Figma, Git, GitHub, HTML, Jira, Pinia, Postman, SASS, TypeScript, Vite, Vue, Vuex
   📄 Requirements excerpt: Your skills & qualifications:

Experience in front end development with HTML, CSS, Sass, Typescript 


Knowledge of modern JS frameworks, preferably V...

📋 Job ID: 3470
   Title: TIGER TRIBE – SENIOR BACKEND DEVELOPER
   Company: TIGER TRIBE
   🛠️  Skills (17): .NET, .NET Core, API Design, ASP.NET, AWS, Azure, C, Cassandra, Communication, GCP, Microservices, MongoDB, NoSQL, PostgreSQL, RESTful API, SQL, SQL Server
   📄 Requirements excerpt: Your skills & qualifications:

5+ years of experience in backend development using .NET technologies (C#, ASP.NET Core, Web API).  


Strong proficien...

📋 Job ID: 3474
   Title: Frontend developer (PA project)
   Company: CUBICASA
   🛠️  Skills (14): CSS, Code Review, F

## 9. Save Backup CSV

In [10]:
from datetime import datetime

backup_file = f'jobs_skills_extracted_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df_updated.to_csv(backup_file, index=False, encoding='utf-8-sig')
print(f'✅ Backup saved: {backup_file}')
print(f'📦 File size: {os.path.getsize(backup_file) / 1024 / 1024:.2f} MB')

try:
    from google.colab import files
    files.download(backup_file)
    print('📥 File downloaded')
except:
    print('💾 File saved locally')

✅ Backup saved: jobs_skills_extracted_20260122_043128.csv
📦 File size: 57.42 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 File downloaded


## 10. Update Database

In [11]:
confirm = input('⚠️  Update database with extracted skills? (yes/no): ')

if confirm.lower() == 'yes':
    print('\n🔄 Updating database...')
    print('⏰ This will update the "updated_at" timestamp for all modified rows\n')
    
    update_count = 0
    error_count = 0
    
    with engine.begin() as conn:
        for idx in tqdm(range(len(df_updated)), desc='Updating database'):
            row = df_updated.iloc[idx]
            
            try:
                update_query = '''
                    UPDATE jobs SET
                        required_skills = :required_skills,
                        updated_at = NOW()
                    WHERE id = :id
                '''
                
                params = {
                    'id': int(row['id']),
                    'required_skills': row['required_skills'] if pd.notna(row['required_skills']) else None
                }
                
                conn.execute(text(update_query), params)
                update_count += 1
                
            except Exception as e:
                error_count += 1
                if error_count <= 5:
                    print(f'\n⚠️ Error updating job {row["id"]}: {str(e)[:100]}')
    
    print(f'\n✅ Updated: {update_count:,} jobs in database')
    print(f'❌ Errors: {error_count:,}')
    
    if error_count == 0:
        # Verify update
        print(f'\n🔍 Verifying database updates...')
        verify_query = 'SELECT COUNT(*) as updated FROM jobs WHERE updated_at >= DATE_SUB(NOW(), INTERVAL 5 MINUTE)'
        result = pd.read_sql(verify_query, engine)
        recently_updated = result['updated'].iloc[0]
        print(f'  ✅ {recently_updated:,} jobs have updated_at within last 5 minutes')
        
        # Show updated counts
        stats_query = '''
            SELECT 
                COUNT(*) as total,
                SUM(CASE WHEN required_skills IS NOT NULL AND required_skills != '[]' THEN 1 ELSE 0 END) as has_skills,
                SUM(CASE WHEN required_skills IS NULL OR required_skills = '[]' THEN 1 ELSE 0 END) as no_skills
            FROM jobs
        '''
        stats = pd.read_sql(stats_query, engine).iloc[0]
        
        print(f'\n📊 Database statistics:')
        print(f'  Total jobs: {stats["total"]:,}')
        print(f'  Has skills: {stats["has_skills"]:,} ({stats["has_skills"]/stats["total"]*100:.1f}%)')
        print(f'  No skills: {stats["no_skills"]:,} ({stats["no_skills"]/stats["total"]*100:.1f}%)')
        
        print(f'\n✅ SUCCESS: Database updated successfully!')
    else:
        print(f'\n⚠️ WARNING: {error_count} errors occurred during update')
else:
    print('❌ Update cancelled by user')


🔄 Updating database...
⏰ This will update the "updated_at" timestamp for all modified rows



Updating database:   0%|          | 0/10044 [00:00<?, ?it/s]


✅ Updated: 10,044 jobs in database
❌ Errors: 0

🔍 Verifying database updates...
  ✅ 1,297 jobs have updated_at within last 5 minutes

📊 Database statistics:
  Total jobs: 12,117.0
  Has skills: 10,176.0 (84.0%)
  No skills: 1,941.0 (16.0%)

✅ SUCCESS: Database updated successfully!


## 11. Cleanup and Summary

In [12]:
# Cleanup temporary SSL files
for f in temp_files:
    try:
        os.unlink(f)
        print(f'🗑️  Deleted temp file: {f}')
    except:
        pass

print('\n' + '='*80)
print('🎉 SKILL EXTRACTION COMPLETE - FINAL SUMMARY')
print('='*80)
print(f'📊 Total jobs processed: {len(df):,}')
print(f'💾 Backup file: {backup_file}')
print(f'⏱️  Processing time: {elapsed_time:.1f} seconds')
print()

extractor.print_stats()
print()

print('📋 TOP 10 MOST IN-DEMAND SKILLS:')
for idx, (skill, count) in enumerate(skill_counts.most_common(10), 1):
    percentage = count / len(df) * 100
    print(f'  {idx}. {skill}: {count:,} jobs ({percentage:.1f}%)')
print()

print('='*80)

🗑️  Deleted temp file: /tmp/tmp7e1swtvx.pem

🎉 SKILL EXTRACTION COMPLETE - FINAL SUMMARY
📊 Total jobs processed: 10,044
💾 Backup file: jobs_skills_extracted_20260122_043128.csv
⏱️  Processing time: 1276.2 seconds

📊 EXTRACTION STATISTICS:
  Processed jobs: 10,044
  Total skills found: 112,759
  Jobs with no skills: 203
  Average skills per job: 11.23

📋 TOP 10 MOST IN-DEMAND SKILLS:
  1. Communication: 4,485 jobs (44.7%)
  2. Python: 3,458 jobs (34.4%)
  3. SQL: 2,541 jobs (25.3%)
  4. Agile: 2,441 jobs (24.3%)
  5. CI/CD: 2,385 jobs (23.7%)
  6. AWS: 2,293 jobs (22.8%)
  7. Training: 2,248 jobs (22.4%)
  8. Java: 2,239 jobs (22.3%)
  9. C: 1,954 jobs (19.5%)
  10. Monitoring: 1,754 jobs (17.5%)

